# Reactive-Transport Model — Coupled Adsorption–Oxidation at Mn-Oxide Interfaces

Companion code for Azimzadeh & Martínez, *Environ. Sci. Technol.* 2025, 59, 19513–19525 
([10.1021/acs.est.5c09711](https://doi.org/10.1021/acs.est.5c09711)).

Workflow: (1) fix the reactor hydraulics from a Br⁻ tracer test, (2) store all fixed
parameters in a config CSV, (3) fit the fully-coupled reactive ODE model — every species
from its own CSV / own time axis, one shared parameter set.


In [ ]:
import os, sys
if not os.path.exists('src'):
    !git clone https://github.com/brz-azimzadeh/Glyphosate-MnOx-Reactive-Transport-Model.git
    %cd Glyphosate-MnOx-Reactive-Transport-Model
!pip install -q -r requirements.txt
sys.path.insert(0, os.getcwd())

In [ ]:
import numpy as np, pandas as pd, functools
from src import (fit_tracer, ExperimentConfig, fit_global_ode,
                 fit_gp_ampa_gly, fit_curve, models, plotting)

## 1. Hydraulic characterization from the Br⁻ tracer (fixed inputs)
Fit the gamma RTD (SI Text S3) to the conservative tracer, then hold the hydraulics
(N, tau, tau_pulse) FIXED. `tau` is the gamma scale; mean hydraulic residence = N·tau.


In [ ]:
br = pd.read_csv('data/Br_tracer.csv', encoding='utf-8-sig')
hyd = fit_tracer(br['Time (min)'], br['[Br] mM'], C0=1.0, t_s=74.0, fix_N=4)
print(hyd.summary())


## 2. Fixed-parameter config (saved to / loaded from CSV)
Hydraulics come from the tracer; `C0`/`t_s` are the experiment conditions; `retardation`
lengthens the step residence for adsorbing species (glyphosate ≈ 2.5; 1.0 = none).
Edit the CSV to model a different reactor or schedule.


In [ ]:
cfg = ExperimentConfig.from_tracer(hyd, C0=300, t_s=160, retardation=2.5)
cfg.to_csv('data/experiment_config.csv')
cfg = ExperimentConfig.from_csv('data/experiment_config.csv')   # round-trip
print('tau_step_eff = %.3f min  (N=%g, tau_pulse=%.2f, C0=%g, t_s=%g)'
      % (cfg.tau_step_eff, cfg.N, cfg.tau_pulse, cfg.C0, cfg.t_s))


## 3. Load each species from its own file
Independent `Time (min)` columns, so measurement intervals can differ per species.


In [ ]:
def load(path, col):
    d = pd.read_csv(path, encoding='utf-8-sig')
    return d['Time (min)'].to_numpy(float), d[col].to_numpy(float)
data = {
    'GP':   load('data/GP.csv',      'GP (uM)'),
    'AMPA': load('data/AMPA.csv',    'AMPA (uM)'),
    'Gly':  load('data/Glycine.csv', 'Glycine (uM)'),
    'Pi':   load('data/Pi.csv',      'Pi (uM)'),
    'NH4':  load('data/NH4.csv',     'NH4+ (uM)'),
    'Mn':   load('data/Mnsoln.csv',  'Mnsoln (uM)'),
}
{sp: len(t) for sp, (t, y) in data.items()}


## 4. Fully-coupled ODE global fit (hydraulics fixed)
One shared parameter set drives every species (Pi←GP,AMPA; NH₄←Gly,AMPA; Mn←GP).
No per-species effective constants, no fixed kinetic values.


In [ ]:
result = fit_global_ode(data, cfg)
print(result.summary())
result.as_dataframe()


Typical per-species R² here: **GP 0.99, AMPA 0.97, Gly 0.93, Mn 0.97**, with
**Pi ≈ 0.60, NH₄ ≈ 0.44**. Pi has no sink in eq S34 (real Pi adsorbs); NH₄ shares
`k_Gly_loss` with glycine — intrinsic costs of full coupling (see README).


## 5. Observed vs. predicted


In [ ]:
t_pred = np.arange(0, 260, 1)
pred = result.predict(t_pred)
obs = {sp: data[sp][1] for sp in ['GP','AMPA','Gly','Pi','NH4']}
plotting.plot_global(data['GP'][0], obs, t_pred,
                     {k: v for k, v in pred.items() if k != 'Mn'},
                     title='Fully-coupled ODE global fit');


## 6. Closed-form engine (the paper's staged method)
The second engine reproduces the published Table 2. It fits GP/AMPA/glycine with one
constrained global fit, then orthophosphate, ammonium, and soluble Mn separately (Pi/NH₄
carry their own effective constants). It uses the **same** tracer-fixed `cfg` as the ODE
engine, but expects the co-gridded species merged into one DataFrame.


In [ ]:
def read_col(path, col):
    d = pd.read_csv(path, encoding='utf-8-sig')
    return d[['Time (min)', col]]

df = functools.reduce(lambda a, b: pd.merge(a, b, on='Time (min)', how='outer'), [
    read_col('data/GP.csv', 'GP (uM)'),       read_col('data/AMPA.csv', 'AMPA (uM)'),
    read_col('data/Glycine.csv', 'Glycine (uM)'), read_col('data/Pi.csv', 'Pi (uM)'),
    read_col('data/NH4.csv', 'NH4+ (uM)')])
mndf = pd.read_csv('data/Mnsoln.csv', encoding='utf-8-sig')
df.shape, mndf.shape

In [ ]:
# GP + AMPA + glycine: one constrained global fit (paper ordering enforced)
gag = fit_gp_ampa_gly(df, cfg)
print(gag.summary())

In [ ]:
# Orthophosphate, ammonium, soluble Mn (fit separately; own effective constants)
pi  = fit_curve(df, cfg, 'Pi', models.make_pi_model(cfg),
                ['k_Gly_form','k_AMPA_form','k_GP_loss','k_AMPA_loss'], p0=[1e-1,1e-1,2e-3,1e-2])
nh4 = fit_curve(df, cfg, 'NH4', models.make_nh4_model(cfg),
                ['k_Gly_form','k_Gly_loss','k_AMPA_form','k_AMPA_loss','k_NH4_loss'],
                p0=[1e-2,1e-2,7e-3,1e-2,5e-3])
mn  = fit_curve(mndf, cfg, 'Mn', models.make_mnsoln_fixed_gp_model(cfg, k_gp_loss=gag.values[0]),
                ['k_Mn_ox'], p0=[1e-2])
print('R2  GP/AMPA/Gly=%.3f  Pi=%.3f  NH4=%.3f  Mn=%.3f' % (gag.r2, pi.r2, nh4.r2, mn.r2))
print('S_AMPA=%.2f (paper 0.33)   t_half=%.1f h (paper 6)' % (gag.extra['S_AMPA'], gag.extra['t_half_GP_h']))

Compare to the published Table 2: k_GP_loss ≈ 1.93×10⁻³ min⁻¹, k_Mn_ox ≈ 1.39×10⁻²,
S_AMPA ≈ 0.35, t½ ≈ 6 h — essentially exact on the well-identified quantities.


In [ ]:
t_cf = np.arange(0, 260, 1)
pred_cf = {}
pred_cf.update(gag.predict(t_cf)); pred_cf.update(pi.predict(t_cf)); pred_cf.update(nh4.predict(t_cf))
obs_cf = {k: df[cfg.columns[k]].to_numpy(float) for k in ['GP','AMPA','Gly','Pi','NH4']}
plotting.plot_global(df['Time (min)'].to_numpy(float), obs_cf, t_cf, pred_cf,
                     title='Closed-form engine (paper staged method)');

## 7. Adapt to your own system
Replace `Br_tracer.csv` and the six species CSVs with your own (any intervals), edit
`experiment_config.csv` (`C0`, `t_s`, `retardation`, …), and re-run. See
`examples/run_tracer_calibration.py` for the scripted version.
